# Pandas for Machine Learning — Part 1 (Concepts & Code)

**Pandas** is the library you'll use to load, explore, clean, and reshape real-world datasets before they ever reach a Machine Learning model. If NumPy gives you fast numeric arrays, Pandas gives you a fast, labeled, spreadsheet-like structure built on top of those arrays — the **DataFrame**.

This is **Part 1** of a two-part Pandas series. It covers everything you need to load a dataset and start exploring, selecting, filtering, and cleaning it.

## Learning Objectives
By the end of this notebook, you will be able to:
- Explain what Pandas is and how it relates to NumPy.
- Create and inspect a `Series` (a single labeled column of data).
- Create a `DataFrame` from dictionaries, lists of dictionaries, and NumPy arrays.
- Read data from a CSV file and explore it with `.head()`, `.tail()`, `.info()`, `.describe()`, `.shape`, `.columns`, `.dtypes`.
- Select columns and rows using `[]`, `.loc[]`, and `.iloc[]`.
- Filter rows using boolean conditions.
- Add, modify, and drop columns.
- Detect and handle missing data (`.isna()`, `.dropna()`, `.fillna()`).
- Sort a DataFrame by one or more columns.

## Prerequisites
- Core Python (variables, lists, dictionaries, loops).
- NumPy basics (`NumPy_Concepts_and_Code.ipynb`) — Pandas is built directly on top of NumPy arrays.
- Pandas is a third-party library. If it isn't already installed, run `pip install pandas` (Google Colab has it pre-installed).

## Companion notebook
After this notebook, open **`Pandas_Practice_Part_1.ipynb`** to test yourself. That notebook contains exercises only — no solutions — so you can practice independently.

---
## Table of Contents
1. [What is Pandas, and Why Does it Matter for ML?](#what-is-pandas)
2. [Importing Pandas](#importing)
3. [The Series](#series)
4. [The DataFrame](#dataframe)
5. [Creating a DataFrame](#creating-dataframe)
6. [Reading Data from a CSV File](#reading-csv)
7. [Exploring a DataFrame](#exploring)
8. [Selecting Columns](#selecting-columns)
9. [Selecting Rows with `.loc[]` and `.iloc[]`](#loc-iloc)
10. [Filtering Rows (Boolean Indexing)](#filtering)
11. [Adding and Modifying Columns](#adding-columns)
12. [Dropping Columns and Rows](#dropping)
13. [Handling Missing Data](#missing-data)
14. [Sorting a DataFrame](#sorting)
15. [Common Mistakes Recap](#common-mistakes)
16. [Summary](#summary)


<a id="what-is-pandas"></a>
# 1. What is Pandas, and Why Does it Matter for ML?

**Pandas** provides two core data structures:
- **`Series`** — a single labeled, one-dimensional array (like one column of a spreadsheet).
- **`DataFrame`** — a labeled, two-dimensional table (like an entire spreadsheet, or a SQL table) made up of multiple Series sharing the same row index.

### Why not just use NumPy arrays directly?
- NumPy arrays are indexed only by position (`0, 1, 2, ...`), and every column must share the same data type.
- Pandas adds **labels** — meaningful row and column names — and lets **different columns hold different types** (e.g. names as text, ages as integers, scores as floats), which is exactly what real-world datasets look like.
- Pandas also brings powerful, convenient tools for reading files, handling missing data, filtering, grouping, and reshaping — tasks that would take many lines of raw NumPy or Python code.

### Where Pandas fits in the ML pipeline
Almost every ML project starts the same way: load a raw dataset (often a CSV file) into a Pandas DataFrame, explore and clean it, then convert the relevant columns into a NumPy array to feed into Scikit-learn. Pandas is the bridge between "messy real-world data" and "clean numeric arrays a model can use."

<a id="importing"></a>
# 2. Importing Pandas

By strong convention, Pandas is always imported with the alias `pd`.

In [ ]:
import pandas as pd
import numpy as np

print(pd.__version__)

<a id="series"></a>
# 3. The Series

A `Series` is a one-dimensional labeled array — think of it as a single column with an index attached.

In [ ]:
marks = pd.Series([78, 85, 91, 67, 88])
print(marks)
print(type(marks))

**Output:**
```
0    78
1    85
2    91
3    67
4    88
dtype: int64
```
The left-hand column (`0, 1, 2, 3, 4`) is the **index** — automatically generated here, but you can supply your own labels.

In [2]:
import pandas as pd
subject_marks = pd.Series([78, 85, 91], index=["Math", "Science", "English"])
print(subject_marks)

print(subject_marks["Science"])   # access by custom label
print(subject_marks.mean())        # Series support the same aggregate methods as NumPy

Math       78
Science    85
English    91
dtype: int64
85
84.66666666666667


### ML Connection
A single column of a real dataset — e.g. every student's "age," or every house's "price" — is exactly a Pandas `Series` once you load the data.

<a id="dataframe"></a>
# 4. The DataFrame

A `DataFrame` is a 2D table: multiple named columns (each a `Series`), sharing a common row index. It's the structure you'll work with almost constantly in Pandas.

In [ ]:
data = {
    "name": ["Ali", "Sara", "Ahmed", "Ayesha"],
    "age": [22, 21, 23, 20],
    "marks": [84, 91, 58, 90],
}

df = pd.DataFrame(data)
print(df)
print(type(df))

**Common mistake:** Confusing a `Series` (one column) with a `DataFrame` (a full table of columns). `df["marks"]` returns a `Series`; `df[["marks"]]` (with double brackets) returns a one-column `DataFrame`. Both are valid, but they behave slightly differently, so keep the distinction in mind.

<a id="creating-dataframe"></a>
# 5. Creating a DataFrame — Other Common Ways

You'll encounter data in several shapes. Pandas can build a DataFrame from all of them.

In [ ]:
# From a list of dictionaries -- one dictionary per row (very common with JSON-style data)
records = [
    {"name": "Ali", "age": 22, "marks": 84},
    {"name": "Sara", "age": 21, "marks": 91},
    {"name": "Ahmed", "age": 23, "marks": 58},
]

df_from_records = pd.DataFrame(records)
print(df_from_records)

In [ ]:
# From a NumPy array, with column names supplied separately
array_data = np.array([
    [78, 85, 90],
    [88, 92, 95],
])

df_from_array = pd.DataFrame(array_data, columns=["Math", "Science", "English"])
print(df_from_array)

### ML Connection
> Dictionaries and lists of dictionaries map naturally to DataFrames — this is exactly the JSON-shaped data you worked with in the Core Python notebooks' serialization sections, and it's often how data arrives from APIs or configuration files.

<a id="reading-csv"></a>
# 6. Reading Data from a CSV File

Real datasets almost always start life as a CSV (comma-separated values) file. `pd.read_csv()` is the single most-used function in all of Pandas. Let's first create a small sample CSV file, then read it back — exactly the round-trip you'll do with real datasets.

In [ ]:
import os

os.makedirs("sample_data", exist_ok=True)

sample_csv_text = """name,age,city,marks
Ali,22,Lahore,84
Sara,21,Karachi,91
Ahmed,23,Lahore,58
Ayesha,20,Islamabad,90
Bilal,24,Karachi,72
"""

with open("sample_data/students.csv", "w") as file:
    file.write(sample_csv_text)

print("students.csv written.")

In [ ]:
df = pd.read_csv("sample_data/students.csv")
print(df)

**Common mistake:** Forgetting that `read_csv` expects a valid file **path**, not raw CSV text. If your notebook can't find the file, double-check the working directory with `os.getcwd()` and confirm the file was actually written first.

<a id="exploring"></a>
# 7. Exploring a DataFrame

Before doing anything else with a new dataset, always inspect it. These are the tools you'll reach for first, every single time.

In [ ]:
df = pd.read_csv("sample_data/students.csv")

print(df.head(3))     # first 3 rows
print()
print(df.tail(2))     # last 2 rows

In [ ]:
print(df.shape)       # (rows, columns)
print(df.columns)     # column names
print(df.dtypes)      # data type of each column

In [ ]:
df.info()             # summary: column names, non-null counts, dtypes, memory usage

In [ ]:
print(df.describe())   # summary statistics for numeric columns: count, mean, std, min, max, quartiles

### ML Connection
`.describe()` is often the very first thing you run on a new dataset — it instantly reveals the scale of each numeric feature, whether there might be outliers (via min/max), and roughly how much data you have (via count).

<a id="selecting-columns"></a>
# 8. Selecting Columns

In [ ]:
df = pd.read_csv("sample_data/students.csv")

print(df["name"])          # single column -> Series
print(type(df["name"]))

print(df[["name", "marks"]])   # multiple columns (double brackets) -> DataFrame
print(type(df[["name", "marks"]]))

<a id="loc-iloc"></a>
# 9. Selecting Rows with `.loc[]` and `.iloc[]`

- **`.loc[]`** — select by **label** (the index value, and column names).
- **`.iloc[]`** — select by **integer position**, just like list/array indexing.

Both accept `[row_selector, column_selector]`.

In [ ]:
df = pd.read_csv("sample_data/students.csv")

print(df.loc[0])                  # row with index label 0 (all columns)
print()
print(df.loc[0, "name"])          # row 0, column 'name'
print()
print(df.loc[0:2, ["name", "marks"]])   # rows 0-2 (INCLUSIVE with .loc), specific columns

In [ ]:
print(df.iloc[0])                 # first row, by position
print()
print(df.iloc[0, 1])               # first row, second column, by position
print()
print(df.iloc[0:2, [0, 3]])        # rows 0-1 (EXCLUSIVE stop, like Python slicing), columns 0 and 3

**Common mistake:** Mixing up `.loc[]` (label-based, and its row-slice **includes** the end label) with `.iloc[]` (position-based, and its row-slice **excludes** the end position) — this is one of the most common Pandas bugs, especially since with a default integer index the two can look deceptively similar.

<a id="filtering"></a>
# 10. Filtering Rows (Boolean Indexing)

Just like NumPy, you can filter a DataFrame's rows using a boolean condition — this builds directly on the boolean indexing you learned in the NumPy notebook.

In [ ]:
df = pd.read_csv("sample_data/students.csv")

mask = df["marks"] >= 80
print(mask)         # a Series of True/False, one per row

print(df[mask])     # only rows where the mask is True

# Usually written in one line:
print(df[df["marks"] >= 80])

In [ ]:
# Combining multiple conditions: use & (and), | (or), and wrap each condition in parentheses
print(df[(df["marks"] >= 70) & (df["city"] == "Karachi")])

**Common mistake:** Using Python's `and`/`or` keywords instead of `&`/`|` for combining conditions on Series, and forgetting the parentheses around each condition. `df["marks"] >= 70 & df["city"] == "Karachi"` will raise a confusing error or give wrong results — always write `(condition1) & (condition2)`.

<a id="adding-columns"></a>
# 11. Adding and Modifying Columns

In [ ]:
df = pd.read_csv("sample_data/students.csv")

df["passed"] = df["marks"] >= 60          # new column, computed from an existing one
print(df)

In [ ]:
df["marks"] = df["marks"] + 5             # modify an existing column (bonus marks)
print(df)

### ML Connection
This is exactly how you'll engineer new **features** for an ML model — e.g. creating a `bmi` column from `weight` and `height` columns, before training a model.

<a id="dropping"></a>
# 12. Dropping Columns and Rows

`.drop()` removes columns or rows. By default it returns a **new** DataFrame — it doesn't modify the original unless you pass `inplace=True` or reassign the result.

In [ ]:
df = pd.read_csv("sample_data/students.csv")

df_no_city = df.drop(columns=["city"])       # drop a column
print(df_no_city)
print()
print(df.columns)   # original DataFrame is untouched

In [ ]:
df_dropped_row = df.drop(index=0)   # drop the row with index label 0
print(df_dropped_row)

**Common mistake:** Calling `df.drop(columns=["city"])` and expecting `df` itself to change. Without `inplace=True` (or reassigning `df = df.drop(...)`), the original DataFrame is left unchanged.

<a id="missing-data"></a>
# 13. Handling Missing Data

Real datasets almost always have missing values, represented in Pandas as `NaN` (Not a Number). Let's create a small dataset with some missing values to practice on.

In [ ]:
messy_data = {
    "name": ["Ali", "Sara", "Ahmed", "Ayesha"],
    "age": [22, np.nan, 23, 20],
    "marks": [84, 91, np.nan, 90],
}

df = pd.DataFrame(messy_data)
print(df)

In [ ]:
print(df.isna())            # True where a value is missing
print()
print(df.isna().sum())      # count of missing values per column

In [ ]:
df_dropped = df.dropna()    # remove any row containing at least one missing value
print(df_dropped)

In [ ]:
df_filled = df.fillna({"age": df["age"].mean(), "marks": df["marks"].mean()})
print(df_filled)   # missing values replaced with each column's own average

**Common mistake:** Calling `df.dropna()` without thinking about how much data you might be discarding — if many rows have at least one missing value, you could lose most of your dataset. Often, `fillna()` with a sensible value (like the column mean) is a gentler alternative, though the right choice always depends on the situation.

### ML Connection
Missing values must be handled **before** training almost any ML model — most algorithms simply cannot process `NaN`. Deciding whether to drop or fill missing data (and with what) is a real, important modeling decision, not just a technical formality.

<a id="sorting"></a>
# 14. Sorting a DataFrame

In [ ]:
df = pd.read_csv("sample_data/students.csv")

print(df.sort_values("marks"))                       # ascending by default
print()
print(df.sort_values("marks", ascending=False))       # descending
print()
print(df.sort_values(["city", "marks"], ascending=[True, False]))  # sort by multiple columns

**Common mistake:** Expecting `df.sort_values(...)` to modify `df` in place. Like `.drop()`, it returns a **new**, sorted DataFrame by default — reassign the result (`df = df.sort_values(...)`) or pass `inplace=True` if you want to keep the change.

<a id="common-mistakes"></a>
# 15. Common Mistakes — Recap

| Mistake | Fix |
|---|---|
| Confusing `df["col"]` (Series) with `df[["col"]]` (single-column DataFrame) | Use double brackets when you specifically need a DataFrame |
| Mixing up `.loc[]` (label-based, inclusive slice end) with `.iloc[]` (position-based, exclusive slice end) | Be deliberate about which one you need |
| Using `and`/`or` instead of `&`/`\|` when combining filter conditions | Always use `&`/`\|` with parentheses around each condition |
| Assuming `.drop()` or `.sort_values()` modifies the DataFrame in place | Reassign the result, or pass `inplace=True` |
| Calling `.dropna()` without checking how much data would be lost | Check `.isna().sum()` first, and consider `.fillna()` as an alternative |

<a id="summary"></a>
# Summary — What You Learned in Part 1

- Pandas' two core structures: `Series` (1D, labeled) and `DataFrame` (2D, labeled table).
- Creating DataFrames from dictionaries, lists of dictionaries, and NumPy arrays.
- Reading real data with `pd.read_csv()`.
- Exploring a DataFrame: `.head()`, `.tail()`, `.shape`, `.columns`, `.dtypes`, `.info()`, `.describe()`.
- Selecting columns (`df["col"]`, `df[["col1","col2"]]`) and rows (`.loc[]`, `.iloc[]`).
- Filtering rows with boolean conditions.
- Adding, modifying, and dropping columns/rows.
- Detecting and handling missing data with `.isna()`, `.dropna()`, `.fillna()`.
- Sorting a DataFrame with `.sort_values()`.

**Next:** Practice these skills in `Pandas_Practice_Part_1.ipynb`, then continue to **Part 2**, which covers `groupby`, merging/joining datasets, the `.apply()`/string/datetime tools, duplicates, and exporting data — finishing with an end-to-end mini analysis.